# model.py walkthrough

Steps through each function in `triage/model.py` on the sample dataset and shows the
outputs, ending with the same report the CLI writes.

## Load and clean

Fixes the hand-entry typos, drops blanks, reports what it did.

In [12]:
import json
import numpy as np
import pandas as pd

from config import TRIAGE_CONFIG
from triage.ingest import ingest
from triage import model

df, quality = ingest('factory_telemetry.csv')
print(json.dumps(quality, indent=2))
df.head()

{
  "rows_loaded": 1000,
  "entries_normalized": 20,
  "rows_excluded": 0,
  "exclusion_reasons": {},
  "rows_analyzed": 1000
}


,unit_id,timestamp,operator_shift_id,sealant_batch_id,dispenser_pressure_psi,nozzle_temp_c,oven_temp_c,pressure_test_result,fail,elapsed_hours
0,UNIT_0001,2026-08-15 06:00:06.094341595,OP_SHIFT_A,BATCH_A101,89.82,44.32,185.00,PASS,0,0.000000
1,UNIT_0002,2026-08-15 06:02:32.000317875,OP_SHIFT_A,BATCH_A101,87.81,44.00,182.75,PASS,0,0.040529
2,UNIT_0003,2026-08-15 06:06:00.609023916,OP_SHIFT_A,BATCH_A101,88.76,45.65,187.86,PASS,0,0.098476
3,UNIT_0004,2026-08-15 06:08:57.211294328,OP_SHIFT_A,BATCH_A101,91.90,45.38,173.81,PASS,0,0.147532
4,UNIT_0005,2026-08-15 06:10:52.179296227,OP_SHIFT_A,BATCH_A101,90.01,42.89,178.94,PASS,0,0.179468


## Raw fail rates

No decisions here, just what looks bad. Note the batch AND two shifts all look guilty -
the tables can't tell cause from coincidence.

In [13]:
obs = model.observe(df)
print(f"Overall fail rate: {obs['overall_rate']:.1%}\n")
print(obs['categorical']['operator_shift_id'])
print()
print(obs['categorical']['sealant_batch_id'])
print('\nnumeric means, FAIL vs PASS:')
for col, m in obs['numeric'].items():
    print(f"  {col:<24} {m['mean_fail']:>8.2f} vs {m['mean_pass']:>8.2f}")

Overall fail rate: 15.3%

                     n  fails    rate  lift_vs_line
operator_shift_id                                  
OP_SHIFT_A         333     10  0.0300        0.1963
OP_SHIFT_B         334     57  0.1707        1.1154
OP_SHIFT_C         332     86  0.2590        1.6930

                    n  fails    rate  lift_vs_line
sealant_batch_id                                  
BATCH_A101        250     13  0.0520        0.3399
BATCH_B102        250    126  0.5040        3.2941
BATCH_C103        250      7  0.0280        0.1830
BATCH_D104        248      7  0.0282        0.1845

numeric means, FAIL vs PASS:
  dispenser_pressure_psi      89.73 vs    89.76
  nozzle_temp_c               44.93 vs    45.07
  oven_temp_c                179.67 vs   180.05
  elapsed_hours               18.64 vs    24.94


## Feature matrix

Telemetry standardized (so coefficients are comparable), categoricals one-hot with
`drop_first` (baseline = BATCH_A101 / OP_SHIFT_A). `elapsed_hours` is left out - it's
nearly collinear with batch identity since drums are consumed sequentially.

In [14]:
X = model._build_features(df)
print(list(X.columns))
X.head()

['dispenser_pressure_psi', 'nozzle_temp_c', 'oven_temp_c', 'operator_shift_id_OP_SHIFT_B', 'operator_shift_id_OP_SHIFT_C', 'sealant_batch_id_BATCH_B102', 'sealant_batch_id_BATCH_C103', 'sealant_batch_id_BATCH_D104']


,dispenser_pressure_psi,nozzle_temp_c,oven_temp_c,operator_shift_id_OP_SHIFT_B,operator_shift_id_OP_SHIFT_C,sealant_batch_id_BATCH_B102,sealant_batch_id_BATCH_C103,sealant_batch_id_BATCH_D104
0,0.020981,-0.479109,1.281782,0.0,0.0,0.0,0.0,0.0
1,-0.639408,-0.688945,0.705662,0.0,0.0,0.0,0.0,0.0
2,-0.327283,0.393024,2.014094,0.0,0.0,0.0,0.0,0.0
3,0.704369,0.215974,-1.583453,0.0,0.0,0.0,0.0,0.0
4,0.083406,-1.416815,-0.269900,0.0,0.0,0.0,0.0,0.0


## Fit

One Logit over everything. Each coefficient = that factor's effect with the rest held
constant. `exp(coef)` is the odds multiplier - B102 is ~14x, the two shifts that looked
terrible above are not significant.

In [15]:
coefs, note = model.fit(df)
table = coefs.copy()
table['odds_ratio'] = np.exp(table['coef'])
table.round(4)

,coef,p_value,odds_ratio
dispenser_pressure_psi,-0.1558,0.1407,0.8558
nozzle_temp_c,-0.0926,0.3693,0.9116
oven_temp_c,-0.1314,0.2202,0.8768
operator_shift_id_OP_SHIFT_B,0.6680,0.1463,1.9503
operator_shift_id_OP_SHIFT_C,0.5089,0.3322,1.6634
sealant_batch_id_BATCH_B102,2.6601,0.0000,14.2972
sealant_batch_id_BATCH_C103,-0.6491,0.1757,0.5225
sealant_batch_id_BATCH_D104,-0.9604,0.0842,0.3827


## Winner

Must be significant, and categorical levels must raise risk (negative dummy coef =
safer than baseline). Largest |coef| among the eligible wins.

In [16]:
features = [model._to_feature(name, row) for name, row in coefs.iterrows()]
alpha = TRIAGE_CONFIG['significance_alpha']

eligible = [f for f in features
            if f.p_value < alpha and (f.kind == 'continuous' or f.coef > 0)]
for f in eligible:
    print(f'{f.name}  ({f.dimension})  coef={f.coef:+.2f}  p={f.p_value:.1e}')

winner = max(eligible, key=lambda f: abs(f.coef))
print(f'\nwinner: {winner.name} -> {winner.dimension}')

BATCH_B102  (MATERIAL)  coef=+2.66  p=4.3e-10

winner: BATCH_B102 -> MATERIAL


## Confidence

Half significance (`-log10(p)`, capped), half coefficient size (bar differs for dummy
vs continuous), capped at 0.95.

In [17]:
winner.confidence = model._score(winner, TRIAGE_CONFIG)

sig = min(1.0, -np.log10(max(winner.p_value, 1e-300)) / TRIAGE_CONFIG['significance_log10_cap'])
effect = min(1.0, abs(winner.coef) / TRIAGE_CONFIG['coef_norm'][winner.kind])
print(f'significance: {sig:.2f}  magnitude: {effect:.2f}  -> confidence {winner.confidence:.2f}')

significance: 0.94  magnitude: 1.00  -> confidence 0.95


## Decoys

Elevated in the raw tables but not confirmed by the model. The check is
`not (significant and positive)` because a pure decoy's coef usually goes negative
once the real cause is in the model.

In [18]:
decoys = []
for f in features:
    if f.kind != 'categorical' or f is winner:
        continue
    if f.p_value < alpha and f.coef > 0:
        continue
    rate_in, _ = model._univariate_rates(obs, f)
    if rate_in > obs['overall_rate']:
        decoys.append(f)
        print(f'{f.name:<12} raw rate {rate_in:.1%}, coef={f.coef:+.2f}, p={f.p_value:.2f}')

OP_SHIFT_B   raw rate 17.1%, coef=+0.67, p=0.15
OP_SHIFT_C   raw rate 25.9%, coef=+0.51, p=0.33


## METHOD check

Time alone looks strongly significant (failures cluster in the bad batch's window).
Modeled next to the winner it collapses - the drift was just the exposure window.

In [19]:
bare = model.time_only_fit(df)
print(f'time alone         : coef={bare.coef:+.2f}, p={bare.p_value:.1e}')

check = model.method_check(df, winner)
print(f'time beside winner : coef={check["coef"]:+.2f}, p={check["p_value"]:.2f}')

time alone         : coef=-0.47, p=3.9e-07
time beside winner : coef=-0.22, p=0.17


## Full run

Same as `python run_triage.py`.

In [20]:
from triage.report import build_report

verdict = model.resolve(df, TRIAGE_CONFIG)
verdict['failure_rate'] = df['fail'].mean()
report = build_report(quality, verdict)
print(json.dumps(report, indent=2))

{
  "total_units_analyzed": 1000,
  "failure_rate_percentage": 15.3,
  "primary_root_cause": {
    "category": "MATERIAL",
    "suspect_attribute": "BATCH_B102",
    "confidence_score": 0.95,
    "summary": "BATCH_B102 is the primary root cause: with every other factor held constant in the regression, it carries the only dominant significant coefficient (+2.66, p = 4.3e-10), raising the odds of a pressure-test failure about 14x. Units using it failed at 50.4% versus 3.6% for everything else. Raw fail-rate tables make OP_SHIFT_B (17.1% fail rate), OP_SHIFT_C (25.9% fail rate) look guilty, but the model does not confirm any of them as a risk factor once BATCH_B102 is accounted for - secondary symptoms, not causes. Modeled alongside BATCH_B102, the time trend collapses to non-significance (p = 0.17), so the apparent drift was the primary cause's exposure window (METHOD ruled out)."
  }
}
